# 03 — Generate Deterministic Test Code from IR
This notebook demonstrates how a structured Intermediate Representation (IR)
can be transformed into deterministic, stable automated test code.

The real code generation engine lives in a private repository.
This notebook uses a simplified, public-safe version to illustrate the concept.


In [1]:
import json
from pathlib import Path


## 1. Load IR from Notebook 02

We load the IR JSON produced earlier.


In [2]:
ir_path = Path("../sample-data/ir_examples/login_ir.json")

with open(ir_path, "r") as f:
    ir = json.load(f)

ir


{'test_name': 'login_valid_credentials',
 'description': 'User logs in with valid credentials',
 'steps': [{'action': 'navigate', 'target': 'login_page'},
  {'action': 'input', 'target': 'username_field', 'value': 'valid_user'},
  {'action': 'input', 'target': 'password_field', 'value': 'valid_password'},
  {'action': 'click', 'target': 'login_button'},
  {'action': 'assert', 'target': 'dashboard_page', 'condition': 'visible'}],
 'metadata': {'priority': 'high', 'tags': ['smoke', 'auth']}}

## 2. Define a Mock Codegen Template

This simulates the deterministic code generation layer.

In the private engine:
- templates are versioned
- selectors are resolved via the mapping layer
- codegen is fully deterministic

Here, we demonstrate the concept with a simplified template.


In [3]:
TEMPLATE_HEADER = """import pytest
from playwright.sync_api import Page, expect

@pytest.mark.smoke
def test_{test_name}(page: Page):
"""

ACTION_MAP = {
    "navigate": '    page.goto("{target}")\n',
    "input": '    page.fill("{target}", "{value}")\n',
    "click": '    page.click("{target}")\n',
    "assert": '    expect(page.locator("{target}")).to_be_visible()\n'
}


## 3. Mock Selector Mapping

In the real system, logical targets (e.g., `login_button`) are mapped to
stable selectors via the application model.

Here we simulate that with a simple dictionary.


In [4]:
SELECTOR_MAP = {
    "login_page": "https://demo-app/login",
    "username_field": "#username",
    "password_field": "#password",
    "login_button": "#login",
    "dashboard_page": "#dashboard"
}


## 4. Generate Code from IR

We iterate through IR steps and fill the template.


In [5]:
def generate_code(ir):
    code = TEMPLATE_HEADER.format(test_name=ir["test_name"])

    for step in ir["steps"]:
        action = step["action"]
        target = SELECTOR_MAP.get(step["target"], step["target"])
        value = step.get("value")

        if action == "input":
            code += ACTION_MAP[action].format(target=target, value=value)
        elif action == "navigate":
            code += ACTION_MAP[action].format(target=target)
        elif action == "click":
            code += ACTION_MAP[action].format(target=target)
        elif action == "assert":
            code += ACTION_MAP[action].format(target=target)
        else:
            code += f"    # Unsupported action: {action}\n"

    return code

generated_code = generate_code(ir)
print(generated_code)


import pytest
from playwright.sync_api import Page, expect

@pytest.mark.smoke
def test_login_valid_credentials(page: Page):
    page.goto("https://demo-app/login")
    page.fill("#username", "valid_user")
    page.fill("#password", "valid_password")
    page.click("#login")
    expect(page.locator("#dashboard")).to_be_visible()



## 5. Save Generated Test File

We write the generated test into `generated-tests-demo/`.


In [6]:
output_path = Path("../generated-tests-demo/test_login_valid_credentials.py")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w") as f:
    f.write(generated_code)

output_path


PosixPath('../generated-tests-demo/test_login_valid_credentials.py')

# Summary

In this notebook, we:

- Loaded IR from Notebook 02
- Used a deterministic template to generate Playwright test code
- Applied a mock selector mapping layer
- Exported a runnable pytest file

Next:
Move to **04_run_tests_and_collect_artifacts.ipynb** to execute the generated test.
